# Xiaohongshu 关键词抓取 Demo（Scrapling）

这个 Notebook 基于 `examples/xhs_keyword_poc.py`，支持：
- 输入单个关键词，或从 CSV 批量读取关键词
- `api` / `render` 双模式
- 关键词之间随机等待 5~6 分钟，降低风控触发概率
- 输出每个关键词单独 CSV + 汇总 CSV

> 说明：若你要求“翻页翻到底”，推荐 `api` 模式（会按 `has_more` 持续翻页到末尾）。

In [ ]:
import random
import time
from pathlib import Path
from typing import List

import pandas as pd

from examples.xhs_keyword_poc import run_api_mode, run_render_mode


In [ ]:
# ===== 用户配置 =====
MODE = 'api'  # 'api' 或 'render'

# 单关键词模式（当 CSV_KEYWORDS_PATH=None 时生效）
SINGLE_KEYWORD = '穿搭'

# CSV 批量关键词模式
# CSV 需至少有一列关键词列，默认列名为 keyword
CSV_KEYWORDS_PATH = None  # 例如: 'keywords.csv'
CSV_KEYWORD_COLUMN = 'keyword'

# 登录态 cookie（原始 Cookie Header 形式）
COOKIE_HEADER = ''

# ===== api 模式参数 =====
API_HEADERS_RAW = '{}'
SEARCH_API_URL = 'https://edith.xiaohongshu.com/api/sns/web/v1/search/notes'
DETAIL_API_URL = 'https://edith.xiaohongshu.com/api/sns/web/v1/feed'
API_PAGE_SIZE = 20

# ===== render 模式参数 =====
# render 无法像 api 那样通过 has_more 严格判定翻页结束，故建议尽量加大 scroll_rounds
RENDER_SCROLL_ROUNDS = 120
RENDER_IDLE_ROUNDS = 3  # 连续N轮无新增note_id即停止滚动

# ===== 通用参数 =====
MAX_NOTES_PER_KEYWORD = 100000  # 给足上限，api 模式会实际翻到底
OUTPUT_DIR = Path('outputs/xhs_demo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_keywords(single_keyword: str, csv_path: str | None, keyword_col: str = 'keyword') -> List[str]:
    if csv_path is None:
        return [single_keyword.strip()] if single_keyword.strip() else []

    df = pd.read_csv(csv_path)
    if keyword_col not in df.columns:
        raise ValueError(f'CSV 中未找到关键词列: {keyword_col}. 可用列: {df.columns.tolist()}')

    kws = (
        df[keyword_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .tolist()
    )
    # 去重但保留顺序
    return list(dict.fromkeys(kws))


def random_sleep_5_to_6_minutes():
    sec = random.uniform(300, 360)
    print(f'⏳ Sleep {sec:.1f}s (~{sec/60:.2f} min) to reduce risk control trigger...')
    time.sleep(sec)


def run_keyword(keyword: str):
    if MODE == 'api':
        return run_api_mode(
            keyword=keyword,
            max_notes=MAX_NOTES_PER_KEYWORD,
            cookie_header=COOKIE_HEADER,
            api_headers_raw=API_HEADERS_RAW,
            search_api_url=SEARCH_API_URL,
            detail_api_url=DETAIL_API_URL,
            page_size=API_PAGE_SIZE,
        )

    return run_render_mode(
        keyword=keyword,
        max_notes=MAX_NOTES_PER_KEYWORD,
        cookie_header=COOKIE_HEADER,
        scroll_rounds=RENDER_SCROLL_ROUNDS,
        render_idle_rounds=RENDER_IDLE_ROUNDS,
    )


In [ ]:
keywords = load_keywords(SINGLE_KEYWORD, CSV_KEYWORDS_PATH, CSV_KEYWORD_COLUMN)
print(f'Loaded {len(keywords)} keywords: {keywords[:10]}')

all_frames = []

for idx, kw in enumerate(keywords, start=1):
    print(f'\n===== [{idx}/{len(keywords)}] keyword={kw} mode={MODE} =====')
    started = time.time()
    items = run_keyword(kw)

    rows = []
    for item in items:
        row = item.__dict__.copy()
        row['keyword'] = kw
        rows.append(row)

    df_kw = pd.DataFrame(rows)
    kw_csv = OUTPUT_DIR / f'{idx:03d}_{kw}_notes.csv'
    df_kw.to_csv(kw_csv, index=False, encoding='utf-8-sig')
    all_frames.append(df_kw)

    print(f'✅ {kw}: {len(df_kw)} rows -> {kw_csv}')
    print(f'⏱ elapsed: {time.time() - started:.1f}s')

    if idx < len(keywords):
        random_sleep_5_to_6_minutes()

if all_frames:
    merged = pd.concat(all_frames, ignore_index=True)
else:
    merged = pd.DataFrame()

merged_csv = OUTPUT_DIR / 'all_keywords_merged.csv'
merged.to_csv(merged_csv, index=False, encoding='utf-8-sig')
print(f'\n🎉 merged rows={len(merged)} -> {merged_csv}')


## CSV 输入示例

`keywords.csv`：

| keyword |
|---|
| 穿搭 |
| 护肤 |
| 旅行攻略 |

将 `CSV_KEYWORDS_PATH = 'keywords.csv'`，并设置 `CSV_KEYWORD_COLUMN = 'keyword'` 即可。